# Chapter 15: Generative AI
**Part IV — Specialist Domains**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

GANs, VAEs, diffusion models, and latent diffusion.

## Learning Objectives
See Chapter 15 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## SPARQL query on a knowledge graph (via SPARQLWrapper)


In [ ]:
# SPARQL query on a knowledge graph (via SPARQLWrapper)
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setQuery("""
    SELECT ?person ?personLabel WHERE {
      ?person wdt:P106 wd:Q82594.   # occupation: computer scientist
      ?person wdt:P27  wd:Q145.     # country of citizenship: UK
      SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
    } LIMIT 10
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()
for r in results["results"]["bindings"]:
    print(r["personLabel"]["value"])

## VAE on 2D toy data — learn the latent structure


In [ ]:
# VAE on 2D toy data — learn the latent structure
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Generate 2-cluster 2D data
torch.manual_seed(42); np.random.seed(42)
n = 500
X1 = torch.randn(n, 2) + torch.tensor([2., 2.])
X2 = torch.randn(n, 2) + torch.tensor([-2., -2.])
X  = torch.cat([X1, X2])
labels = np.array([0]*n + [1]*n)

class VAE(nn.Module):
    def __init__(self, input_dim=2, latent_dim=2, hidden=32):
        super().__init__()
        self.enc_mu  = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, latent_dim))
        self.enc_lv  = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden), nn.ReLU(), nn.Linear(hidden, input_dim))
    
    def reparameterise(self, mu, log_var):
        std = torch.exp(0.5 * log_var)
        return mu + std * torch.randn_like(std)
    
    def forward(self, x):
        mu, lv = self.enc_mu(x), self.enc_lv(x)
        z = self.reparameterise(mu, lv)
        return self.decoder(z), mu, lv

vae = VAE(); opt = optim.Adam(vae.parameters(), lr=1e-3)
losses = []
for epoch in range(500):
    recon, mu, lv = vae(X)
    recon_loss = ((recon - X)**2).sum(dim=1).mean()
    kl_loss    = -0.5 * (1 + lv - mu**2 - lv.exp()).sum(dim=1).mean()
    loss = recon_loss + kl_loss
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

# Visualise
with torch.no_grad():
    _, mu, _ = vae(X)
    z = mu.numpy()
    samples = vae.decoder(torch.randn(200, 2)).numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2E75B6' if l==0 else '#C0392B' for l in labels]
axes[0].scatter(X[:,0], X[:,1], c=colors, alpha=0.4, s=15)
axes[0].set_title('Original data (2 clusters)'); axes[0].grid(alpha=0.3)
axes[1].scatter(z[:,0], z[:,1], c=colors, alpha=0.4, s=15)
axes[1].set_title('Latent space (encoder mean μ)'); axes[1].grid(alpha=0.3)
axes[2].scatter(samples[:,0], samples[:,1], c='#1F7D6E', alpha=0.5, s=15, label='Generated')
axes[2].scatter(X[:,0].numpy(), X[:,1].numpy(), c='gray', alpha=0.15, s=8, label='Real')
axes[2].set_title('Generated samples (from N(0,I))'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.suptitle('Variational Autoencoder (VAE) on 2D toy data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ch15_vae.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 📚 Review Questions

See Chapter 15 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 15 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*